# EDEN-01 — Climate Data Engineering (ETL) · v1.1
**Project:** Environmental Digital Twin for Ecosystem Intelligence and Navigation (EDEN)  
**Phase:** M1 — Data engineering | Notebook 01 of 15 (16 total) | **CPU only**

## Mission
Turn verified raw climate/control/weather observations from EDEN-00 into consistent, timestamped, documented tables. ETL means **Extract → Transform → Load**. We will preserve raw values, identify missing readings, align measured weather without using future observations, and define reproducible data splits before fitting anything.

`EDEN-00 → EDEN-01 (this notebook) → EDEN-02 RGB-D acquisition → EDEN-07 alignment → forecasting / physics / fusion`

## Run on Kaggle
1. Save the successful EDEN-00 version. Open a fresh Kaggle notebook and import this file.
2. Keep your attached **Greenhouse Digital Twin** dataset. This version accepts its automatically extracted `eden/raw/` folder, an intact EDEN-00 output, or a ZIP containing that output. No new download is required. Do not attach the 31 GB camera archive yet.
3. Accelerator: None. Internet: off is fine. Run cells in order. Save Version → Save & Run All after PASS.
4. If multiple EDEN-00 outputs are attached, set `UPSTREAM` in Cell 1 to the exact folder containing its `manifest.json` or the exact ZIP path.

## Handoff files (under `/kaggle/working/eden/`)
- `data/climate.parquet`: unique compartment/time keys, climate measurements, measured weather, missingness and QC flags, split labels.
- `data/controls.parquet`: realized actuator-related measurements and separate setpoints; never confuse the two.
- `data/weather.parquet`: observed weather on a regular UTC grid; `data/weather_forecast.parquet`: archived forecast, quarantined from training until issue-time semantics are known.
- `data/raw_tables/*.parquet`: full source columns and original timestamps; `data/Harvest.xlsx`: original manual harvest workbook (not a predictor).
- `data/splits.json`, `data/schema_map.json`: frozen split protocol and exact metadata-backed mappings.
- `reports/`: source profiles, QC summaries, plots, harvest sheet inventory; `manifest.json`: checksums and provenance.

**Scope:** no imputation, scaling, neural training or crop image download. No new scientific performance claim. Parquet uses compression and preserves column types. The canonical files retain missing values rather than inventing data.

## Cell 1 — Initialize the environment
Paths separate read-only inputs from writable outputs. The optional environment variables are for local testing; Kaggle users normally leave them unchanged. We use Pandas for tables, NumPy for numeric checks, Matplotlib for diagnostics, and PyArrow (via Pandas) for Parquet. No GPU needed.

In [ ]:
from pathlib import Path
import os, io, json, hashlib, zipfile, shutil, importlib.util
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

INPUT = Path(os.environ.get('EDEN_INPUT', '/kaggle/input'))  # Read-only upstream files
ROOT = Path(os.environ.get('EDEN_OUTPUT', '/kaggle/working/eden'))
UPSTREAM = None  # Optional: Path('/kaggle/input/.../eden') or Path('/kaggle/input/.../results.zip')
DATA = ROOT / 'data'
REPORTS = ROOT / 'reports'
RAW_TABLES = DATA / 'raw_tables'
for p in (DATA, REPORTS, RAW_TABLES):
    p.mkdir(parents=True, exist_ok=True)
assert shutil.disk_usage(ROOT).free > 200 * 2**20, 'Need at least 200 MiB free'
print('Pandas:', pd.__version__, '| Output:', ROOT)
print('PyArrow available:', importlib.util.find_spec('pyarrow') is not None)

def save_json(path, obj):
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding='utf-8')

def sha(data):
    return hashlib.sha256(data).hexdigest()

## Cell 2 — Locate and verify EDEN-00 (ZIP or extracted folder)
Kaggle may automatically unpack ZIP files during dataset upload. We first verify the seven other EDEN-00 artifacts against their manifest. If the numeric ZIP exists, we also verify its original SHA-256 and publisher MD5. If the ZIP is missing, we find the genuine extracted files by their original relative paths (allowing an extra outer folder) and verify their byte counts and SHA-256 fingerprints. The fingerprints below were calculated from your uploaded, verified EDEN-00 `results.zip`, not from guessed filenames. A missing, altered or ambiguous file stops execution. We do not recreate the original compressed archive or claim to have verified it on Kaggle when it is absent. The original archive hash is recorded as a reference, with the actual verification mode documented. No operating-system metadata is required.


In [ ]:
def open_upstream(path):
    path = Path(path)
    if path.is_dir():
        return lambda name: (path / name).read_bytes()
    if path.suffix.lower() == '.zip':
        with zipfile.ZipFile(path) as z:
            names = [n for n in z.namelist() if n.endswith('manifest.json') and Path(n).name == 'manifest.json']
            matches = [n for n in names if json.loads(z.read(n)).get('producer') == 'EDEN-00']
            assert len(matches) == 1, 'Expected exactly one EDEN-00 manifest in ZIP'
            prefix = matches[0][:-len('manifest.json')]
        def read_member(name):
            with zipfile.ZipFile(path) as z:
                return z.read(prefix + name)
        return read_member
    raise ValueError('UPSTREAM must be a folder or ZIP')

if UPSTREAM is None:
    candidates = []
    for p in sorted(INPUT.rglob('manifest.json')):
        try:
            if json.loads(p.read_text()).get('producer') == 'EDEN-00':
                candidates.append(p.parent)
        except (ValueError, OSError):
            continue
    for p in sorted(INPUT.rglob('*.zip')):
        try:
            reader = open_upstream(p)
            if json.loads(reader('manifest.json')).get('producer') == 'EDEN-00':
                candidates.append(p)
        except (AssertionError, ValueError, OSError, zipfile.BadZipFile, KeyError):
            continue
    assert len(candidates) == 1, f'Attach one EDEN-00 output or set UPSTREAM. Found: {candidates}'
    UPSTREAM = candidates[0]
read_upstream = open_upstream(UPSTREAM)
upstream_manifest_bytes = read_upstream('manifest.json')
upstream_manifest = json.loads(upstream_manifest_bytes)
DOI = '10.4121/fa102772-32db-4b30-bace-12f2016722ce.v1'
assert upstream_manifest['producer'] == 'EDEN-00'
assert upstream_manifest['source_doi'] == DOI
assert upstream_manifest['csv_parse_errors'] == 0
ARCHIVE_REL = 'raw/autonomous_greenhouse_challenge4_timeseries.zip'
# Reference fingerprints from the verified original archive in the user's EDEN-00 output.
SOURCE_ARCHIVE_SHA256 = '19bfddf994ffeeb6eff8c20da14af8ae09956d23d7c6d85d7e5aa017cff3f7b1'
MEMBER_FINGERPRINTS = {'autonomous_greenhouse_challenge4_timeseries/Harvest.xlsx': {'sha256': '72e28dea530f957f47bbfcfb3a51202be52fbfe6681dbedc669d4fbb8fa38765',
                                                              'size': 77244},
 'autonomous_greenhouse_challenge4_timeseries/ReadMe.md': {'sha256': '846383812e764a2f0f879181fa0164e6963be43420eb662c11dcc2f250c02555',
                                                           'size': 2540},
 'autonomous_greenhouse_challenge4_timeseries/channel_info.json': {'sha256': 'a0801761154ef82f45ebb82a17810eac972d15b50ff53bafd3a0335fcdca2ed7',
                                                                   'size': 144102},
 'autonomous_greenhouse_challenge4_timeseries/timeseries/agrifusion.csv': {'sha256': '6e19963780ed0b534cdeadd0d9e30c85967d49d19304abb0225f5d88ee9aa314',
                                                                           'size': 9213579},
 'autonomous_greenhouse_challenge4_timeseries/timeseries/ideas.csv': {'sha256': '5f828a34a781961226a990393196843e6fec3c9b71f663312a7f0507c66eefda',
                                                                      'size': 7845268},
 'autonomous_greenhouse_challenge4_timeseries/timeseries/mugrow.csv': {'sha256': 'ef49f8065f5220e31e6697f467ca985b6c938c72c4a5ee2d533beb274ee4598d',
                                                                       'size': 8409907},
 'autonomous_greenhouse_challenge4_timeseries/timeseries/reference.csv': {'sha256': '2aad86c83fececf647b746daaecee1087cf41d8828aa8d3de060c15f08c51cd8',
                                                                          'size': 7181925},
 'autonomous_greenhouse_challenge4_timeseries/timeseries/tomatonuts.csv': {'sha256': 'e0f2d974dca226657a302b04383afcd8b62fb2e44830c793edfd9f7a6415cd60',
                                                                           'size': 8810999},
 'autonomous_greenhouse_challenge4_timeseries/timeseries/trigger.csv': {'sha256': '0e9ba8bd7b9a39f8b2666571796d8452d7e5eb2a6c5e3d5a7d7e2c0260b9fa7b',
                                                                        'size': 9724872},
 'autonomous_greenhouse_challenge4_timeseries/timeseries/weather.csv': {'sha256': '56c2a6b6d78d6ff7c9747766196da84ba634676a7e783964db793320e4d1d0fa',
                                                                        'size': 2032859},
 'autonomous_greenhouse_challenge4_timeseries/timeseries/weather_forecast.csv': {'sha256': 'bc828b7de0b3408f4fd56c6c9f7d3117a083a643c5989a60d867128f2d187d52',
                                                                                 'size': 892232}}

entries = {e['path']:e for e in upstream_manifest['files']}
assert len(entries) == len(upstream_manifest['files']), 'Duplicate manifest paths'
assert ARCHIVE_REL in entries, 'Original archive missing from EDEN-00 manifest'
assert entries[ARCHIVE_REL]['sha256'] == SOURCE_ARCHIVE_SHA256, 'Unexpected source archive version'
assert entries[ARCHIVE_REL]['size'] == 6014133
verified_upstream = []
for name, entry in entries.items():
    if name == ARCHIVE_REL:
        continue  # Verify the archive separately, or verify its extracted contents instead.
    payload = read_upstream(name)
    assert len(payload) == entry['size'] and sha(payload) == entry['sha256'], name
    verified_upstream.append(name)
source_manifest = json.loads(read_upstream('source_manifest.json'))

try:
    raw_zip = read_upstream(ARCHIVE_REL)
except (FileNotFoundError, KeyError):
    raw_zip = None  # Missing ZIP is expected for Kaggle's extracted dataset layout.

member_bytes = {}
member_paths = {}
if raw_zip is not None:
    assert len(raw_zip) == 6014133 and sha(raw_zip) == SOURCE_ARCHIVE_SHA256
    assert hashlib.md5(raw_zip).hexdigest() == 'c4da1b2696c4a1e5ea5744f2482b0f93'
    with zipfile.ZipFile(io.BytesIO(raw_zip)) as z:
        assert z.testzip() is None
        for name in MEMBER_FINGERPRINTS:
            member_bytes[name] = z.read(name)
            member_paths[name] = name
    verified_upstream.append(ARCHIVE_REL)
    source_mode = 'original_zip'
else:
    assert Path(UPSTREAM).is_dir(), 'Extracted mode requires an attached directory'
    raw_dir = Path(UPSTREAM) / 'raw'
    assert raw_dir.is_dir(), f'Missing raw folder: {raw_dir}'
    # Allow one or more enclosing folders added by Kaggle, but match original path suffixes.
    available = [p for p in raw_dir.rglob('*') if p.is_file()
                 and '__MACOSX' not in p.parts and not p.name.startswith(('._','.DS_Store'))]
    for name in MEMBER_FINGERPRINTS:
        parts = Path(name).parts
        candidates = [p for p in available if p.relative_to(raw_dir).parts[-len(parts):] == parts]
        assert len(candidates) == 1, f'Expected one extracted file for {name}; found {candidates}'
        p = candidates[0]
        member_bytes[name] = p.read_bytes()
        member_paths[name] = str(p.relative_to(Path(UPSTREAM)))
    source_mode = 'extracted_members'

for name, spec in MEMBER_FINGERPRINTS.items():
    data = member_bytes[name]
    assert len(data) == spec['size'] and sha(data) == spec['sha256'], f'File differs from original: {name}'

source_verification = dict(mode=source_mode, original_archive_sha256=SOURCE_ARCHIVE_SHA256,
    original_archive_verified_here=(raw_zip is not None), verified_upstream_files=verified_upstream,
    verified_members=[dict(path=name, input_path=member_paths[name], **spec) for name,spec in MEMBER_FINGERPRINTS.items()])
save_json(REPORTS/'source_verification.json',source_verification)
print('PASS: upstream', UPSTREAM)
print('Input mode:', source_mode)
print('Manifest files verified:', len(verified_upstream), '/', len(entries))
print('Original data members SHA-256 verified:', len(member_bytes), '/', len(MEMBER_FINGERPRINTS))


## Cell 3 — Extract all genuine tables and publisher metadata
The archive contains six compartment CSVs, observed weather, archived weather forecast, a channel dictionary and a harvest workbook. We explicitly exclude macOS metadata. `channel_info.json` is the authority for units, measurement versus parameter semantics, and the Europe/Amsterdam timezone. Team IDs come from the source README. All original columns are retained separately, even those not yet used in EDEN's main model.
The same verified bytes are used in both input modes; subsequent ETL is identical. The harvest workbook is read as an original `.xlsx` file, not the individual sheet previews shown in Kaggle.


In [ ]:
TEAMS = {'tomatonuts':'3.01', 'mugrow':'3.02', 'agrifusion':'3.03',
         'reference':'3.06', 'ideas':'3.07', 'trigger':'3.08'}
EXPERIMENT = 'AGC4-2024'
EXPECTED = set(TEAMS) | {'weather', 'weather_forecast'}

def genuine(name):
    p = Path(name)
    return '__MACOSX' not in p.parts and not p.name.startswith(('._', '.DS_Store'))

members = [n for n in member_bytes if genuine(n)]
csv_members = {Path(n).stem:n for n in members if n.endswith('.csv')}
assert set(csv_members) == EXPECTED, csv_members
raw = {k:pd.read_csv(io.BytesIO(member_bytes[n]), encoding='utf-8-sig', low_memory=False) for k,n in csv_members.items()}
channel_info = json.loads(member_bytes[next(n for n in members if Path(n).name == 'channel_info.json')])
harvest_bytes = member_bytes[next(n for n in members if Path(n).name == 'Harvest.xlsx')]
archive_readme = member_bytes[next(n for n in members if Path(n).name.lower() == 'readme.md')].decode('utf-8-sig')
assert channel_info['timezone'] == 'Europe/Amsterdam'
assert set(channel_info['components']) == EXPECTED
profiles = pd.DataFrame([{'table':k,'rows':len(v),'source_columns':len(v.columns)} for k,v in raw.items()])
display(profiles)
print('Total compartment rows:', sum(len(raw[t]) for t in TEAMS))
print('Source timezone:', channel_info['timezone'])
print(archive_readme)

## Cell 4 — Parse timestamps and measure gaps
A timestamp with a UTC offset identifies a specific instant. `pd.to_datetime(..., utc=True)` converts the source offset to UTC, including the October daylight-saving change; do not strip the offset. We retain the original timestamp and sort by UTC. Duplicate keys must be resolved explicitly: exact duplicate rows can be dropped, conflicting observations cause an error. We inspect interval lengths and require timestamps on the 5-minute grid. No interpolation occurs.

In [ ]:
STEP = pd.Timedelta(minutes=5)
parsed = {}
quality = []
for team, frame in raw.items():
    d = frame.copy()
    assert d['time'].astype(str).str.contains(r'(?:Z|[+-]\d{2}:\d{2})$').all(), team
    d['ts_utc'] = pd.to_datetime(d['time'], utc=True, errors='raise')
    assert d['ts_utc'].notna().all(), team
    duplicates = int(d['ts_utc'].duplicated().sum())
    d = d.drop_duplicates().sort_values('ts_utc').reset_index(drop=True)
    assert d['ts_utc'].is_unique, f'Conflicting duplicate timestamps: {team}'
    assert (d['ts_utc'].dt.floor('5min') == d['ts_utc']).all(), team
    gaps = d['ts_utc'].diff().dropna()
    missing_ticks = int(((gaps / STEP) - 1).sum())
    quality.append(dict(table=team, source_rows=len(frame), unique_rows=len(d), duplicate_keys=duplicates,
                        missing_5min_ticks=missing_ticks, start_utc=str(d.ts_utc.min()), end_utc=str(d.ts_utc.max())))
    parsed[team] = d
    for idx in d.index[d['ts_utc'].diff() > STEP]:
        print('GAP:', team, d.loc[idx-1,'ts_utc'], '→', d.loc[idx,'ts_utc'])
quality = pd.DataFrame(quality)
display(quality)

## Cell 5 — Approve exact metadata-backed schema mappings
EDEN-00 proposed regex candidates; here we replace them with exact source names and verify each source unit and type against publisher metadata for every team. Climate measurements are targets; actuator-related realized states and controller setpoints are distinct. Pipe temperature is a heating proxy, not power. Irrigation flow duration is not water volume. CO₂ actuation codes are preserved as 1/2 (the source says “1 = on, 2 = out”; no undocumented conversion). Percent values remain 0–100. PAR is photon flux, not W/m². Optional variables absent from a team would be recorded as missing; these selected core channels are required and mismatches stop execution.

In [ ]:
def flatten(node, prefix=''):
    out = {prefix + k:v for k,v in node.get('datastreams', {}).items()}
    for k,v in node.get('components', {}).items():
        out.update(flatten(v, prefix + k + '/'))
    return out
META = {k:flatten(v) for k,v in channel_info['components'].items()}
# canonical name: (original name, publisher unit, publisher type)
CLIMATE = {
 'air_temp_c': ('compartment/air_temperature','degree_Celsius','measurement'),
 'rh_pct': ('compartment/relative_humidity','percent','measurement'),
 'co2_ppm': ('compartment/co2_concentration','ppm','measurement'),
 'par_inside_umol_m2_s': ('compartment/par','micromole / meter ** 2 / second','measurement'),
 'humidity_deficit_g_m3': ('compartment/humidity_deficit','gram / meter ** 3','measurement'),
}
CONTROLS = {
 'vent_lee_pct': ('compartment/window_position_lee_side','percent','measurement'),
 'vent_wind_pct': ('compartment/window_position_wind_side','percent','measurement'),
 'heating_pipe_temp_c': ('compartment/heating_lower_circuit/pipe_temperature','degree_Celsius','measurement'),
 'lighting_pct': ('compartment/lamps_activation_percentage','percent','measurement'),
 'co2_actuation_code': ('compartment/co2_actuation_state','dimensionless','measurement'),
 'irrigation_flow_duration_min': ('compartment/water_supply/water_flow_duration','minute','measurement'),
 'energy_screen_pct': ('compartment/screen_energy/screen_position','percent','measurement'),
 'blackout_screen_pct': ('compartment/screen_blackout/screen_position','percent','measurement'),
 'heating_setpoint_c': ('compartment/heating_temperature_setpoint','degree_Celsius','parameter'),
 'ventilation_setpoint_c': ('compartment/ventilation_temperature_setpoint','degree_Celsius','parameter'),
 'lighting_setpoint_pct': ('compartment/lamps_activation_percentage_setpoint','percent','parameter'),
 'co2_setpoint_ppm': ('compartment/co2_concentration_setpoint','ppm','parameter'),
 'irrigation_interval_setpoint_min': ('compartment/water_supply/water_supply_interval_setpoint','minute','parameter'),
}
WEATHER = {
 'outside_temp_c': ('weather/air_temperature.outside','degree_Celsius','measurement'),
 'outside_rh_pct': ('weather/relative_humidity.outside','percent','measurement'),
 'outside_vapor_g_m3': ('weather/air_absolute_humidity_content.outside','gram / meter ** 3','measurement'),
 'radiation_w_m2': ('weather/radiation_global','watt / meter ** 2','measurement'),
 'wind_speed_m_s': ('weather/wind_speed','meter / second','measurement'),
 'outside_par_umol_m2_s': ('weather/par.outside','micromole / meter ** 2 / second','measurement'),
}
review = []
for team in TEAMS:
    for canonical,(source,unit,kind) in {**CLIMATE, **CONTROLS}.items():
        assert source in parsed[team], (team,source,'missing column')
        m = META[team][source]
        assert (m['unit'],m['type']) == (unit,kind), (team,source,m)
        review.append(dict(table=team,canonical=canonical,source=source,unit=unit,kind=kind,description=m.get('description','')))
for canonical,(source,unit,kind) in WEATHER.items():
    assert source in parsed['weather']
    m = META['weather'][source]
    assert (m['unit'],m['type']) == (unit,kind), (source,m)
    review.append(dict(table='weather',canonical=canonical,source=source,unit=unit,kind=kind,description=m.get('description','')))
review = pd.DataFrame(review)
display(review[review.table.isin(['agrifusion','weather'])])
print('PASS: metadata mappings:', len(review))

## Cell 6 — Transform climate and controls into consistent tables
We use a common composite key `(experiment_id, compartment_id, ts_utc)` to distinguish different compartments at the same time. Every team gets a regular five-minute grid within its own observed span (no artificial extension after the trial ends). Missing source rows stay missing; an explicit flag marks them. Numeric conversion fails on unexpected nonnumeric strings. We retain source timestamps in the raw tables and add observed masks to every canonical signal. No filling or normalization is performed.

In [ ]:
KEYS = ['experiment_id','compartment_id','ts_utc']

def canonical_grid(frame, mapping):
    index = pd.date_range(frame.ts_utc.min(), frame.ts_utc.max(), freq='5min', tz='UTC')
    original = frame.set_index('ts_utc')
    out = pd.DataFrame(index=index)
    out.index.name = 'ts_utc'
    out['source_row_present'] = out.index.isin(original.index)
    for target,(source,_,_) in mapping.items():
        values = pd.to_numeric(original[source], errors='raise').astype(float)
        assert not np.isinf(values.to_numpy()).any(), source
        out[target] = values.reindex(index)
        out[target + '_observed'] = out[target].notna()
    return out.reset_index()

climates, control_tables = [], []
for team, compartment in TEAMS.items():
    for mapping, target in [(CLIMATE,climates),(CONTROLS,control_tables)]:
        d = canonical_grid(parsed[team], mapping)
        d['experiment_id'] = EXPERIMENT
        d['compartment_id'] = compartment
        d['team'] = team
        target.append(d)
climate = pd.concat(climates, ignore_index=True).sort_values(KEYS).reset_index(drop=True)
controls = pd.concat(control_tables, ignore_index=True).sort_values(KEYS).reset_index(drop=True)
assert climate[KEYS].equals(controls[KEYS])
weather = canonical_grid(parsed['weather'], WEATHER).rename(columns={'source_row_present':'weather_source_row_present'})
weather['experiment_id'] = EXPERIMENT
print('Canonical climate rows:', len(climate), '| controls rows:', len(controls))
display(climate.head(3))

## Cell 7 — Align measured weather without leakage
An exact timestamp join uses only weather measured at the same UTC instant. Missing weather remains NaN, including the missing daylight-saving hour and the first climate tick. No nearest-neighbor or backward fill is used. Weather forecast is kept separate: the dataset does not yet establish forecast issuance times, so it cannot be assumed available in an operational forecast. Future measured weather may only be used later in a clearly labeled known-weather experiment, never as future operational truth.

In [ ]:
climate = climate.merge(weather, on=['experiment_id','ts_utc'], how='left', validate='many_to_one')
weather_flags = ['weather_source_row_present'] + [k+'_observed' for k in WEATHER]
for c in weather_flags:
    climate[c] = climate[c].eq(True)
weather_forecast = parsed['weather_forecast'].copy()
weather_forecast['experiment_id'] = EXPERIMENT
print('Climate rows after weather alignment:', len(climate))
print('Missing outside temperature:', climate.outside_temp_c.isna().sum())
print('Archived forecast quarantined:', len(weather_forecast), 'rows')

## Cell 8 — Data quality checks and masks
Missing values are not zeros. We count missing observations, flag conservative engineering plausibility bounds, and keep every original measurement. Bounds below are **project QC heuristics**, not crop recommendations or proof of sensor failure. The Reference compartment's −10°C reading is flagged for review, not silently removed. `*_observed` describes availability; `*_qc_ok` describes finite, in-range readings. Future training should exclude suspect targets by using these masks. We do not automatically remove or repair any row.

In [ ]:
LIMITS = {'air_temp_c':(0,60), 'rh_pct':(0,100), 'co2_ppm':(0,5000),
          'par_inside_umol_m2_s':(0,3000), 'humidity_deficit_g_m3':(0,100),
          'outside_temp_c':(-40,60), 'outside_rh_pct':(0,100), 'outside_vapor_g_m3':(0,100),
          'radiation_w_m2':(0,1500), 'wind_speed_m_s':(0,100), 'outside_par_umol_m2_s':(0,4000)}
CONTROL_LIMITS = {k:(0,100) for k in CONTROLS if k.endswith('_pct')}
CONTROL_LIMITS.update({'heating_pipe_temp_c':(0,120), 'irrigation_flow_duration_min':(0,1440)})
for col,(lo,hi) in LIMITS.items():
    climate[col+'_qc_ok'] = climate[col].notna() & climate[col].between(lo,hi)
for col in CONTROLS:
    if col in CONTROL_LIMITS:
        lo,hi = CONTROL_LIMITS[col]
        controls[col+'_qc_ok'] = controls[col].notna() & controls[col].between(lo,hi)
    elif col == 'co2_actuation_code':
        controls[col+'_qc_ok'] = controls[col].isin([1,2])
    else:
        controls[col+'_qc_ok'] = controls[col].notna()  # Availability only; no invented bounds
climate['target_triplet_valid'] = climate[['air_temp_c_qc_ok','rh_pct_qc_ok','co2_ppm_qc_ok']].all(axis=1)
qc_rows = []
for label, df, fields in [('climate',climate,{**CLIMATE,**WEATHER}),('controls',controls,CONTROLS)]:
    for team,g in df.groupby('team'):
        for c in fields:
            qc_rows.append(dict(table=label,team=team,field=c,rows=len(g),missing=int(g[c].isna().sum()),
              suspect_observed=int((g[c].notna() & ~g[c+'_qc_ok']).sum()),min=g[c].min(),max=g[c].max()))
qc = pd.DataFrame(qc_rows)
display(qc[qc.field.isin(['air_temp_c','rh_pct','co2_ppm','outside_temp_c'])])
suspect = climate.loc[climate.air_temp_c.notna() & ~climate.air_temp_c_qc_ok, KEYS+['team','air_temp_c']]
display(suspect.head(20))

## Cell 9 — Freeze two independent evaluation protocols
**Primary:** hold out whole compartments to test generalization. We select the two highest compartment IDs deterministically (Trigger test; IDEAS validation); the other four train. This is a design decision, not a model-performance result. Because we already inspected all source quality, this is a prospective modeling holdout, not a completely unseen data audit. Shared weather is a common covariate, not independent weather across compartments.

**Secondary:** chronological blocked splits inside each compartment, with fixed UTC boundaries and a 12-hour embargo after each cutoff (6h lookback + 6h horizon). These are separate experiments: never combine the primary and chronological labels in one evaluation. Future windows must have all input and target ticks within the same split and compartment; scalers/imputers/physics fit on training only. EDEN-02 should use the primary compartment split for imagery and preserve time labels for secondary tests. No random row split.

In [ ]:
PRIMARY = {'train':['3.01','3.02','3.03','3.06'], 'validation':['3.07'], 'test':['3.08']}
role = {c:s for s,ids in PRIMARY.items() for c in ids}
assert len(role) == len(TEAMS)
CUT1 = pd.Timestamp('2024-10-15T00:00:00Z')
CUT2 = pd.Timestamp('2024-11-01T00:00:00Z')
EMBARGO = pd.Timedelta(hours=12)
for df in (climate, controls):
    df['split'] = df.compartment_id.map(role)
    t = df.ts_utc
    df['chronological_split'] = np.select([t < CUT1, (t >= CUT1+EMBARGO)&(t < CUT2), t >= CUT2+EMBARGO],
                                          ['train','validation','test'], default='embargo')
SPLITS = dict(schema_version='1.0', experiment_id=EXPERIMENT, seed=42,
              primary={'kind':'held_out_compartment','compartments':PRIMARY,'selection':'highest two compartment IDs, fixed before model fitting'},
              chronological={'kind':'blocked_time','train_end_exclusive':CUT1.isoformat(),
               'validation_start_inclusive':(CUT1+EMBARGO).isoformat(),'validation_end_exclusive':CUT2.isoformat(),
               'test_start_inclusive':(CUT2+EMBARGO).isoformat(),'embargo_minutes':720},
              max_lookback_steps=72,max_horizon_steps=72,timestep_minutes=5,
              window_rule='All input and target timestamps must be in one compartment and one split; exclude embargo; fit preprocessing on training only.',
              primary_column='split',chronological_column='chronological_split',
              source_archive_sha256=SOURCE_ARCHIVE_SHA256, source_verification_mode=source_mode, original_archive_verified_here=(raw_zip is not None), upstream_manifest_sha256=sha(upstream_manifest_bytes))
split_counts = climate.groupby(['team','split','chronological_split']).size().rename('rows').reset_index()
display(split_counts)

## Cell 10 — Explore the greenhouse without training
Line charts show climate trajectories and the relationship between outside weather and internal conditions. The plots are descriptive QC, not model fitting. Daily means omit missing values; the original five-minute records remain unchanged. The missingness chart reveals sensor availability. These plots must not be used to tune models on held-out performance.

In [ ]:
fig, axes = plt.subplots(3,1,figsize=(13,10),sharex=True)
for team,g in climate.groupby('team'):
    daily = g.set_index('ts_utc')[['air_temp_c','rh_pct','co2_ppm']].resample('D').mean()
    for ax,c in zip(axes,['air_temp_c','rh_pct','co2_ppm']):
        ax.plot(daily.index,daily[c],label=team,linewidth=1)
for ax, title in zip(axes,['Air temperature (°C)','Relative humidity (%)','CO₂ (ppm)']):
    ax.set_ylabel(title); ax.grid(alpha=.25)
axes[0].legend(ncol=3,fontsize=8)
axes[-1].set_xlabel('UTC date')
fig.suptitle('EDEN-01 — daily observed climate means (QC only)')
fig.tight_layout(); fig.savefig(REPORTS/'climate_daily.png',dpi=150); plt.show(); plt.close(fig)
missing = climate.groupby('team')[[k+'_observed' for k in [*CLIMATE,*WEATHER]]].mean()
fig, ax = plt.subplots(figsize=(13,4))
im = ax.imshow((1-missing).to_numpy()*100,aspect='auto',cmap='viridis')
ax.set_xticks(range(len(missing.columns)),[c.replace('_observed','') for c in missing.columns],rotation=45,ha='right')
ax.set_yticks(range(len(missing.index)),missing.index)
fig.colorbar(im,ax=ax,label='Missing (%)'); fig.tight_layout()
fig.savefig(REPORTS/'missingness.png',dpi=150); plt.show(); plt.close(fig)

## Cell 11 — Save the data contract and artifacts
Parquet stores typed columns efficiently. We preserve every source table separately (including original time strings and added UTC timestamps), save the canonical tables, and keep the harvest workbook as a separate label source. The workbook is inventoried without joining it to climate: its measurement timing and labels must be reviewed later. The schema JSON contains the metadata-backed mapping and limitations. If PyArrow is unavailable in Kaggle, install it in an internet-enabled session, then rerun this cell; do not substitute a CSV with a `.parquet` extension.

In [ ]:
assert importlib.util.find_spec('pyarrow') is not None, 'PyArrow required for Parquet. Install pyarrow then rerun.'
for name,d in parsed.items():
    d.to_parquet(RAW_TABLES / f'{name}.parquet',index=False,engine='pyarrow',compression='snappy')
for name,d in [('climate',climate),('controls',controls),('weather',weather),('weather_forecast',weather_forecast)]:
    d.to_parquet(DATA/f'{name}.parquet',index=False,engine='pyarrow',compression='snappy')
(DATA/'Harvest.xlsx').write_bytes(harvest_bytes)
with pd.ExcelFile(io.BytesIO(harvest_bytes),engine='openpyxl') as book:
    harvest_inventory = [{'sheet':s,'rows':len(d),'columns':len(d.columns)} for s in book.sheet_names for d in [book.parse(s)]]
save_json(REPORTS/'harvest_inventory.json',harvest_inventory)
save_json(DATA/'channel_info.json',channel_info)
save_json(DATA/'schema_map.json',dict(schema_version='1.0',source_doi=DOI,source_timezone=channel_info['timezone'],
    canonical_timezone='UTC',experiment_id=EXPERIMENT,compartments=TEAMS,fields=review.to_dict('records'),
    qc_bounds=LIMITS,control_qc_bounds=CONTROL_LIMITS,notes=['Exact units/types verified against metadata.',
    'No imputation, scaling, or removal of suspect readings.', 'Weather forecast issuance time unknown; separate table only.',
    'Harvest workbook not joined; no labels inferred.', 'CO2 actuation 1/2 retained; pipe temperature and irrigation duration are proxies.']))
save_json(DATA/'splits.json',SPLITS)
profiles.to_csv(REPORTS/'source_profiles.csv',index=False)
quality.to_csv(REPORTS/'timestamp_quality.csv',index=False)
qc.to_csv(REPORTS/'signal_quality.csv',index=False)
review.to_csv(REPORTS/'schema_review.csv',index=False)
split_counts.to_csv(REPORTS/'split_counts.csv',index=False)
suspect.to_csv(REPORTS/'suspect_air_temperature.csv',index=False)
(ROOT/'source_manifest.json').write_bytes(read_upstream('source_manifest.json'))
(ROOT/'LICENSES.txt').write_text('Source: Maree et al. (2025), 4th Autonomous Greenhouse Challenge: Dwarf Tomato Timeseries and Images, v1, 4TU.ResearchData. DOI: '+DOI+'\nCC BY 4.0: https://creativecommons.org/licenses/by/4.0/\nEDEN modifications: timestamp conversion, schema mapping, regular grids, QC flags and split labels.\n',encoding='utf-8')
print('Saved data and reports to', ROOT)

## Cell 12 — Acceptance gate and handoff manifest
Assertions enforce unique keys, UTC timestamps, complete grid intervals, correct weather alignment, split disjointness, consistent missingness flags and Parquet readback. The manifest records SHA-256 for the produced files and links to EDEN-00. It is written only after validation. A PASS certifies ETL and software consistency, not greenhouse model accuracy or final image validation.

In [ ]:
for df in (climate,controls):
    assert not df.duplicated(KEYS).any()
    assert str(df.ts_utc.dt.tz) == 'UTC'
    assert df.split.notna().all()
    for _,g in df.groupby('compartment_id'):
        assert g.ts_utc.diff().dropna().eq(STEP).all()
    for s,ids in PRIMARY.items():
        assert set(df.loc[df.split.eq(s),'compartment_id']) == set(ids)
    assert set(df.chronological_split) == {'train','validation','test','embargo'}
    assert df.loc[df.chronological_split.eq('train'),'ts_utc'].max() < CUT1
    assert df.loc[df.chronological_split.eq('validation'),'ts_utc'].min() >= CUT1+EMBARGO
    assert df.loc[df.chronological_split.eq('validation'),'ts_utc'].max() < CUT2
    assert df.loc[df.chronological_split.eq('test'),'ts_utc'].min() >= CUT2+EMBARGO
assert climate[KEYS+['split','chronological_split']].equals(controls[KEYS+['split','chronological_split']])
assert weather.ts_utc.is_unique and weather.ts_utc.diff().dropna().eq(STEP).all()
for df,fields in [(climate,{**CLIMATE,**WEATHER}),(controls,CONTROLS)]:
    for c in fields:
        assert df[c+'_observed'].eq(df[c].notna()).all(), c
        assert not (df[c+'_qc_ok'] & ~df[c+'_observed']).any(), c
assert not any(c.startswith('weather_forecast/') for c in climate.columns)
# Reproduce the exact observed-weather join independently from original data.
for c,(source,_,_) in WEATHER.items():
    expected = climate.ts_utc.map(parsed['weather'].set_index('ts_utc')[source]).astype(float)
    assert np.allclose(climate[c],expected,equal_nan=True), c
for name,expected in [('climate',climate),('controls',controls),('weather',weather),('weather_forecast',weather_forecast)]:
    saved = pd.read_parquet(DATA/f'{name}.parquet',engine='pyarrow')
    pd.testing.assert_frame_equal(saved,expected,check_dtype=True)
for name,expected in parsed.items():
    pd.testing.assert_frame_equal(pd.read_parquet(RAW_TABLES/f'{name}.parquet'),expected,check_dtype=True)
report = f"""# EDEN-01 ETL report
- Source: {DOI}; upstream manifest SHA-256: {sha(upstream_manifest_bytes)}
- Source verification mode: {source_mode}; original archive verified here: {raw_zip is not None}; all {len(member_bytes)} genuine member hashes verified.
- Compartment source rows: {sum(len(raw[t]) for t in TEAMS)}; canonical rows: {len(climate)}
- Weather grid rows: {len(weather)}; missing weather grid ticks: {int((~weather.weather_source_row_present).sum())}
- Suspect observed air-temperature rows: {len(suspect)} (retained and flagged)
- Valid climate target triplets: {int(climate.target_triplet_valid.sum())}
- No future weather forecast joined; no imputation or scaling.
- Primary and chronological split protocols frozen; image acquisition pending.
- PASS: ETL acceptance and Parquet readback.
"""
(REPORTS/'etl_report.md').write_text(report,encoding='utf-8')
files = []
for p in sorted(ROOT.rglob('*')):
    if p.is_file() and p != ROOT/'manifest.json':
        b = p.read_bytes()
        files.append(dict(path=str(p.relative_to(ROOT)),size=len(b),sha256=sha(b)))
manifest = dict(schema_version='1.0',producer='EDEN-01',code_version='1.1',source_doi=DOI,
    upstream_manifest_sha256=sha(upstream_manifest_bytes),source_archive_sha256=SOURCE_ARCHIVE_SHA256, source_verification_mode=source_mode, original_archive_verified_here=(raw_zip is not None),
    generated_utc=datetime.now(timezone.utc).isoformat(),status='PASS_ETL',schema_review='selected_fields_metadata_checked',
    image_inventory='deferred_to_EDEN-02',files=files)
save_json(ROOT/'manifest.json',manifest)
print(report)
print('PASS: EDEN-01 climate ETL | tracked outputs:',len(files))

## Next step — EDEN-02 RGB-D pilot
Save the completed version and verify `eden/manifest.json`, `eden/data/splits.json`, and `eden/data/climate.parquet` appear in the Kaggle Output tab. Share the output ZIP or executed notebook. EDEN-02 will attach both EDEN-00 source information and EDEN-01 split definitions, investigate selective remote ZIP access, then validate a small paired RGB/depth sample across compartments without downloading the entire archive blindly.

**Limitations:** In extracted mode the original compressed ZIP is not present; its SHA-256 is a provenance reference, while every genuine member is independently SHA-256 verified against the original. If Kaggle has converted or changed files, verification fails; attach the original notebook output instead. QC thresholds are screening heuristics; no sensor fault is proven. Weather issue-time semantics and harvest labels are unresolved. Full SRS data gate still needs image pairs and calibration verification. The models must not train on unreviewed channels or use held-out outcomes for preprocessing.

**Attribution:** Maree, S.; Zhang, P.; van Marrewijk, B. M.; de Zwart, H. F.; Bijlaard, M.; Hemming, S. (2025). *4th Autonomous Greenhouse Challenge: Dwarf Tomato Timeseries and Images*, v1. 4TU.ResearchData. https://doi.org/10.4121/fa102772-32db-4b30-bace-12f2016722ce.v1. CC BY 4.0.